Если окружение ещё не настроено:

```python
# %pip install pandas pyarrow numpy matplotlib seaborn scikit-learn jupyter ipykernel
```


In [1]:
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
TRACK = "solo"  # "solo" or "team"
VALID_DAYS = 1
MAX_TRAIN_ROWS = 3_500_000
RANDOM_STATE = 42
BLEND_GRID_STEP = 0.05 

LAGS_30M = [1, 2, 3, 6, 12, 24, 48]
ROLL_WINDOWS = [2, 4, 8, 16, 48]

TRACK_CONFIG = {
    "solo": {
        "train_path": "train_solo_track.parquet",
        "test_path": "test_solo_track.parquet",
        "target_col": "target_1h",
        "forecast_points": 8,
    },
    "team": {
        "train_path": "train_team_track.parquet",
        "test_path": "test_team_track.parquet",
        "target_col": "target_2h",
        "forecast_points": 10,
    },
}

CONFIG = TRACK_CONFIG[TRACK]
TARGET_COL = CONFIG["target_col"]
FORECAST_POINTS = CONFIG["forecast_points"]
FUTURE_TARGET_COLS = [f"target_step_{step}" for step in range(1, FORECAST_POINTS + 1)]

ENSEMBLE_MODEL_SPECS = [
    {
        "name": "lgb_poisson_7d",
        "kind": "lgbm",
        "enabled": True,
        "train_days": 7,
        "decay_days": 3.0,
        "params": dict(
            objective="poisson",
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=200,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
    {
        "name": "lgb_poisson_9d",
        "kind": "lgbm",
        "enabled": True,
        "train_days": 9,
        "decay_days": 4.0,
        "params": dict(
            objective="poisson",
            n_estimators=1700,
            learning_rate=0.025,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=250,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_alpha=1.5,
            reg_lambda=4.0,
            random_state=RANDOM_STATE + 17,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
    {
        "name": "lgb_mae_7d",
        "kind": "lgbm",
        "enabled": True,
        "train_days": 7,
        "decay_days": 3.0,
        "params": dict(
            objective="mae",
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=200,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE + 31,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
    {
        "name": "lgb_poisson_5d",
        "kind": "lgbm",
        "enabled": True,
        "train_days": 5,
        "decay_days": 2.0,
        "params": dict(
            objective="poisson",
            n_estimators=1400,
            learning_rate=0.035,
            num_leaves=127,
            max_depth=-1,
            min_child_samples=120,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE + 71,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
    {
        "name": "lgb_poisson_14d",
        "kind": "lgbm",
        "enabled": True,
        "train_days": 14,
        "decay_days": 6.0,
        "params": dict(
            objective="poisson",
            n_estimators=1900,
            learning_rate=0.025,
            num_leaves=255,
            max_depth=-1,
            min_child_samples=120,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE + 91,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
    {
        "name": "lgb_mae_14d",
        "kind": "lgbm",
        "enabled": True,
        "train_days": 14,
        "decay_days": 6.0,
        "params": dict(
            objective="mae",
            n_estimators=2900,
            learning_rate=0.025,
            num_leaves=255,
            max_depth=-1,
            min_child_samples=120,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE + 91,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
    {
        "name": "cat_poisson_14d",
        "kind": "catboost",
        "enabled": True,
        "train_days": 14,
        "decay_days": 6.0,
        "params": dict(
            loss_function="MAE",
            eval_metric="MAE",
            has_time=True,
            iterations=2000,
            learning_rate=0.03,
            depth=8,
            l2_leaf_reg=5.0,
            min_data_in_leaf=100,
            random_seed=RANDOM_STATE + 211,
            verbose=False,
            allow_writing_files=False,
        ),
    },
    {
    "name": "lgb_poisson_7d_chain",
    "kind": "lgbm_chain",
    "enabled": False,
    "train_days": 7,
    "decay_days": 3.0,
    "chain_depth": 3,  
    "params": dict(
        objective="poisson",
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=200,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=3.0,
        random_state=RANDOM_STATE + 101,
        n_jobs=-1,
        verbosity=-1,
    ),
},
    {
        "name": "ridge_7d",
        "kind": "ridge",
        "enabled": True,
        "train_days": 7,
        "decay_days": None,
        "alpha": 4.0,
        "max_train_rows": 3_500_000,
    },
    {
        "name": "ridge_7d_alpha_20",
        "kind": "ridge",
        "enabled": True,
        "train_days": 7,
        "decay_days": None,
        "alpha": 20.0,
        "max_train_rows": 3_500_000,
    },
]

ACTIVE_MODEL_SPECS = [spec for spec in ENSEMBLE_MODEL_SPECS if spec.get("enabled", True)]
MAX_MODEL_TRAIN_DAYS = max(spec["train_days"] for spec in ACTIVE_MODEL_SPECS)
print("Active models:", [spec["name"] for spec in ACTIVE_MODEL_SPECS])


Active models: ['lgb_poisson_7d', 'lgb_poisson_9d', 'lgb_mae_7d', 'lgb_poisson_5d', 'lgb_poisson_14d', 'lgb_mae_14d', 'cat_poisson_14d', 'ridge_7d', 'ridge_7d_alpha_20']


## Загрузка данных


In [19]:
train_df = pd.read_parquet(CONFIG["train_path"])
test_df = pd.read_parquet(CONFIG["test_path"])

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print("track:", TRACK)
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)


track: solo
train shape: (4630000, 9)
test shape: (8000, 3)


In [20]:
display(train_df.head())
display(test_df.head())


,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,target_1h
0,0,2025-07-28 00:00:00,13,13,7,58053,136443,97,87359
1,0,2025-07-28 00:30:00,6,18,13,58867,134643,95,182291
2,0,2025-07-28 01:00:00,7,8,27,54848,135534,96,271432
3,0,2025-07-28 01:30:00,7,8,5,53086,136497,100,169736
4,0,2025-07-28 02:00:00,1,6,0,57561,135314,101,38880


,id,route_id,timestamp
0,3920,0,2025-11-01 11:00:00
1,3921,0,2025-11-01 11:30:00
2,3922,0,2025-11-01 12:00:00
3,3923,0,2025-11-01 12:30:00
4,3924,0,2025-11-01 13:00:00


In [21]:
print("Train date range:", train_df["timestamp"].min(), "->", train_df["timestamp"].max())
print("Test date range:", test_df["timestamp"].min(), "->", test_df["timestamp"].max())
print("Train routes:", train_df["route_id"].nunique())
print("Test routes:", test_df["route_id"].nunique())


Train date range: 2025-07-28 00:00:00 -> 2025-11-01 10:30:00
Test date range: 2025-11-01 11:00:00 -> 2025-11-01 14:30:00
Train routes: 1000
Test routes: 1000


In [22]:
status_cols = sorted([col for col in train_df.columns if col.startswith("status_")])
print("Status columns:", status_cols)
print("Target column:", TARGET_COL)
print("Forecast points:", FORECAST_POINTS)


Status columns: ['status_1', 'status_2', 'status_3', 'status_4', 'status_5', 'status_6']
Target column: target_1h
Forecast points: 8


## Генерируем будущие таргеты


In [23]:
route_group = train_df.groupby("route_id", sort=False)

for step in range(1, FORECAST_POINTS + 1):
    train_df[f"target_step_{step}"] = route_group[TARGET_COL].shift(-step)

train_df[["route_id", "timestamp", TARGET_COL] + FUTURE_TARGET_COLS].head(10)


,route_id,timestamp,target_1h,target_step_1,target_step_2,target_step_3,target_step_4,target_step_5,target_step_6,target_step_7,target_step_8
0,0,2025-07-28 00:00:00,87359,182291.0,271432.0,169736.0,38880.0,134137.0,134137.0,77760.0,77760.0
1,0,2025-07-28 00:30:00,182291,271432.0,169736.0,38880.0,134137.0,134137.0,77760.0,77760.0,76829.0
2,0,2025-07-28 01:00:00,271432,169736.0,38880.0,134137.0,134137.0,77760.0,77760.0,76829.0,193469.0
3,0,2025-07-28 01:30:00,169736,38880.0,134137.0,134137.0,77760.0,77760.0,76829.0,193469.0,194401.0
4,0,2025-07-28 02:00:00,38880,134137.0,134137.0,77760.0,77760.0,76829.0,193469.0,194401.0,99590.0
5,0,2025-07-28 02:30:00,134137,134137.0,77760.0,77760.0,76829.0,193469.0,194401.0,99590.0,138470.0
6,0,2025-07-28 03:00:00,134137,77760.0,77760.0,76829.0,193469.0,194401.0,99590.0,138470.0,116641.0
7,0,2025-07-28 03:30:00,77760,77760.0,76829.0,193469.0,194401.0,99590.0,138470.0,116641.0,38880.0
8,0,2025-07-28 04:00:00,77760,76829.0,193469.0,194401.0,99590.0,138470.0,116641.0,38880.0,38880.0
9,0,2025-07-28 04:30:00,76829,193469.0,194401.0,99590.0,138470.0,116641.0,38880.0,38880.0,77760.0


In [ ]:
import gc
import numpy as np

HISTORY_STEPS = max(max(LAGS_30M), max(ROLL_WINDOWS))
BUFFER_DAYS = int(np.ceil(HISTORY_STEPS / 48)) + 1
RAW_WINDOW_DAYS = MAX_MODEL_TRAIN_DAYS + VALID_DAYS + BUFFER_DAYS

raw_start = train_df["timestamp"].max() - pd.Timedelta(days=RAW_WINDOW_DAYS)

work_df = train_df.loc[train_df["timestamp"] >= raw_start].copy()

for col in status_cols + [TARGET_COL] + FUTURE_TARGET_COLS:
    if col in work_df.columns:
        work_df[col] = work_df[col].astype("float32")

work_df["route_id"] = work_df["route_id"].astype("category")

print("work_df shape:", work_df.shape)
print("work_df range:", work_df["timestamp"].min(), "->", work_df["timestamp"].max())

# исходный широкий train_df больше не нужен
del train_df
gc.collect()


work_df shape: (817000, 17)
work_df range: 2025-10-15 10:30:00 -> 2025-11-01 10:30:00


6835

In [ ]:
# ФИЧИ: время + лаги + роллинги + агрегаты статусов
base_signal_cols = status_cols + [TARGET_COL]

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hour"] = df["timestamp"].dt.hour.astype("int8")
    df["minute"] = df["timestamp"].dt.minute.astype("int8")
    df["dayofweek"] = df["timestamp"].dt.dayofweek.astype("int8")
    df["is_weekend"] = (df["dayofweek"] >= 5).astype("int8")
    df["half_hour_idx"] = (df["hour"] * 2 + df["minute"] // 30).astype("int8")

    df["half_hour_sin"] = np.sin(2 * np.pi * df["half_hour_idx"] / 48)
    df["half_hour_cos"] = np.cos(2 * np.pi * df["half_hour_idx"] / 48)
    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    return df

def add_status_aggregates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    current_cols = [c for c in ["status_1", "status_2", "status_3"] if c in df.columns]
    prev_cols = [c for c in ["status_4", "status_5", "status_6"] if c in df.columns]

    df["status_current_sum"] = df[current_cols].sum(axis=1)
    df["status_prev_sum"] = df[prev_cols].sum(axis=1)
    df["status_total_sum"] = df[current_cols + prev_cols].sum(axis=1)
    df["status_prev_to_current_ratio"] = df["status_prev_sum"] / (df["status_current_sum"] + 1.0)
    return df

def add_wb_promo_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    date = df["timestamp"].dt.normalize()
    years = sorted(df["timestamp"].dt.year.unique())

    # ---------- 11.11 ----------
    # Публично подтвержденное окно на WB Guru:
    # 27 октября 00:00 -> 13 ноября 23:59
    wb_1111_starts = pd.to_datetime([f"{y}-10-27" for y in years])
    wb_1111_ends = pd.to_datetime([f"{y}-11-13" for y in years])
    wb_1111_day = pd.to_datetime([f"{y}-11-11" for y in years])

    df["is_1111_sale_window"] = 0
    for start, end in zip(wb_1111_starts, wb_1111_ends):
        df["is_1111_sale_window"] |= ((date >= start) & (date <= end))

    # days_to_1111
    date_vals = date.values.astype("datetime64[D]")
    days_to_1111 = np.full(len(df), 9999, dtype=np.int16)

    for ev in wb_1111_day.values.astype("datetime64[D]"):
        diff = (ev - date_vals).astype("timedelta64[D]").astype(np.int16)
        days_to_1111 = np.minimum(days_to_1111, diff)

    df["is_pre_1111_14d"] = ((days_to_1111 >= 1) & (days_to_1111 <= 14)).astype("int8")

    df["days_to_1111_clip14"] = np.where(
        (days_to_1111 >= 1) & (days_to_1111 <= 14),
        days_to_1111,
        0
    ).astype("int8")

    df["proximity_to_1111"] = np.where(
        (days_to_1111 >= 1) & (days_to_1111 <= 14),
        15 - days_to_1111,
        0
    ).astype("int8")

    df["is_1111_sale_window"] = df["is_1111_sale_window"].astype("int8")

    # ---------- День рождения WB ----------
    # Гипотеза по мотивам окна 2024: 7-20 октября
    df["is_wb_birthday_window_oct"] = (
        (df["timestamp"].dt.month == 10) &
        (df["timestamp"].dt.day >= 7) &
        (df["timestamp"].dt.day <= 20)
    ).astype("int8")

    # ---------- Back-to-school ----------
    # Грубая сезонная эвристика: вторая половина августа до 1 сентября
    df["is_back_to_school_season"] = (
        ((df["timestamp"].dt.month == 8) & (df["timestamp"].dt.day >= 15)) |
        ((df["timestamp"].dt.month == 9) & (df["timestamp"].dt.day == 1))
    ).astype("int8")

    df["is_back_to_school_peak"] = (
        (df["timestamp"].dt.month == 8) &
        (df["timestamp"].dt.day >= 20) &
        (df["timestamp"].dt.day <= 31)
    ).astype("int8")

    # ---------- Общий агрегат ----------
    df["is_promo_like_period"] = (
        (df["is_1111_sale_window"] == 1) |
        (df["is_wb_birthday_window_oct"] == 1) |
        (df["is_back_to_school_season"] == 1)
    ).astype("int8")

    # Небольшой числовой прокси интенсивности
    df["promo_score"] = (
        5 * df["is_1111_sale_window"] +
        2 * df["is_pre_1111_14d"] +
        2 * df["is_wb_birthday_window_oct"] +
        1 * df["is_back_to_school_peak"]
    ).astype("int8")

    return df

def add_route_seasonal_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    route_hh_target_mean = (
        df.groupby(["route_id", "half_hour_idx"], observed=False)[TARGET_COL]
          .mean()
          .astype("float32")
          .rename("route_hh_target_mean")
          .reset_index()
    )

    route_dow_hh_target_mean = (
        df.groupby(["route_id", "dayofweek", "half_hour_idx"], observed=False)[TARGET_COL]
          .mean()
          .astype("float32")
          .rename("route_dow_hh_target_mean")
          .reset_index()
    )

    route_dow_hh_current_mean = (
        df.groupby(["route_id", "dayofweek", "half_hour_idx"], observed=False)["status_current_sum"]
          .mean()
          .astype("float32")
          .rename("route_dow_hh_current_mean")
          .reset_index()
    )

    route_dow_hh_prev_mean = (
        df.groupby(["route_id", "dayofweek", "half_hour_idx"], observed=False)["status_prev_sum"]
          .mean()
          .astype("float32")
          .rename("route_dow_hh_prev_mean")
          .reset_index()
    )

    df = df.merge(route_hh_target_mean, on=["route_id", "half_hour_idx"], how="left")
    df = df.merge(route_dow_hh_target_mean, on=["route_id", "dayofweek", "half_hour_idx"], how="left")
    df = df.merge(route_dow_hh_current_mean, on=["route_id", "dayofweek", "half_hour_idx"], how="left")
    df = df.merge(route_dow_hh_prev_mean, on=["route_id", "dayofweek", "half_hour_idx"], how="left")

    df["target_vs_route_hh_mean"] = (df[TARGET_COL] / (df["route_hh_target_mean"] + 1.0)).astype("float32")
    df["target_vs_route_dow_hh_mean"] = (df[TARGET_COL] / (df["route_dow_hh_target_mean"] + 1.0)).astype("float32")

    return df

def add_global_timestamp_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    global_ts = (
        df.groupby("timestamp", observed=False)
          .agg(
              global_current_sum=("status_current_sum", "sum"),
              global_prev_sum=("status_prev_sum", "sum"),
              global_target_sum=(TARGET_COL, "sum"),
          )
          .astype("float32")
          .reset_index()
    )

    df = df.merge(global_ts, on="timestamp", how="left")

    df["route_share_current"] = (
        df["status_current_sum"] / (df["global_current_sum"] + 1.0)
    ).astype("float32")

    df["route_share_prev"] = (
        df["status_prev_sum"] / (df["global_prev_sum"] + 1.0)
    ).astype("float32")

    df["route_share_target"] = (
        df[TARGET_COL] / (df["global_target_sum"] + 1.0)
    ).astype("float32")

    return df

def add_lag_features(df: pd.DataFrame, group_col: str = "route_id") -> pd.DataFrame:
    df = df.copy()
    grp = df.groupby(group_col, sort=False)

    for col in base_signal_cols:
        for lag in LAGS_30M:
            df[f"{col}_lag_{lag}"] = grp[col].shift(lag).astype("float32")

        prev = grp[col].shift(1)
        for window in ROLL_WINDOWS:
            rolled = (
                prev.groupby(df[group_col], observed=False)
                    .rolling(window=window, min_periods=1)
                    .mean()
                    .reset_index(level=0, drop=True)
                    .astype("float32")
            )
            df[f"{col}_rollmean_{window}"] = rolled

        df[f"{col}_diff_1"] = grp[col].diff(1).astype("float32")
        df[f"{col}_diff_2"] = grp[col].diff(2).astype("float32")

    return df

work_df = add_time_features(work_df)
# work_df = add_wb_promo_features(work_df)
work_df = add_status_aggregates(work_df)
work_df = add_route_seasonal_features(work_df)
work_df = add_global_timestamp_features(work_df)
work_df = add_lag_features(work_df)

print("work_df after features:", work_df.shape)
display(work_df.head())

/var/folders/l8/dyh502jx1tj9m0wk6bhm75dr0000gn/T/ipykernel_14777/3639097142.py:186: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grp = df.groupby(group_col, sort=False)


work_df after features: (817000, 140)


/var/folders/l8/dyh502jx1tj9m0wk6bhm75dr0000gn/T/ipykernel_14777/3639097142.py:203: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_diff_1"] = grp[col].diff(1).astype("float32")
/var/folders/l8/dyh502jx1tj9m0wk6bhm75dr0000gn/T/ipykernel_14777/3639097142.py:204: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_diff_2"] = grp[col].diff(2).astype("float32")


,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,target_1h,target_step_1,...,target_1h_lag_12,target_1h_lag_24,target_1h_lag_48,target_1h_rollmean_2,target_1h_rollmean_4,target_1h_rollmean_8,target_1h_rollmean_16,target_1h_rollmean_48,target_1h_diff_1,target_1h_diff_2
0,0,2025-10-15 10:30:00,8.0,20.0,31.0,47969.0,175451.0,17.0,209791.0,270379.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2025-10-15 11:00:00,5.0,20.0,28.0,47463.0,183100.0,15.0,270379.0,194968.0,...,NaN,NaN,NaN,209791.0,209791.0,209791.0,209791.0,209791.0,60588.0,NaN
2,0,2025-10-15 11:30:00,17.0,9.0,13.0,45579.0,176317.0,15.0,194968.0,194968.0,...,NaN,NaN,NaN,240085.0,240085.0,240085.0,240085.0,240085.0,-75411.0,-14823.0
3,0,2025-10-15 12:00:00,11.0,11.0,23.0,46175.0,175948.0,17.0,194968.0,119435.0,...,NaN,NaN,NaN,232673.5,225046.0,225046.0,225046.0,225046.0,0.0,-75411.0
4,0,2025-10-15 12:30:00,7.0,9.0,1.0,45085.0,174355.0,18.0,119435.0,80555.0,...,NaN,NaN,NaN,194968.0,217526.5,217526.5,217526.5,217526.5,-75533.0,-75533.0


## Подготовка train и test


In [26]:
feature_cols = [
    col for col in work_df.columns
    if col not in {"timestamp", "id", *FUTURE_TARGET_COLS}
]

target_ready_mask = work_df[FUTURE_TARGET_COLS].notna().all(axis=1)

train_model_df = work_df.loc[
    target_ready_mask,
    feature_cols + ["timestamp"] + FUTURE_TARGET_COLS
].copy()

train_model_df = train_model_df.rename(columns={"timestamp": "source_timestamp"})

print("train_model_df shape:", train_model_df.shape)
print("train_model_df range:", train_model_df["source_timestamp"].min(), "->", train_model_df["source_timestamp"].max())

train_model_df shape: (809000, 140)
train_model_df range: 2025-10-15 10:30:00 -> 2025-11-01 06:30:00


In [27]:
train_ts_max = train_model_df["source_timestamp"].max()
train_window_start = train_ts_max - pd.Timedelta(days=MAX_MODEL_TRAIN_DAYS)
train_model_df = train_model_df[train_model_df["source_timestamp"] >= train_window_start].copy()

print("Rows kept for ensemble window:", train_model_df.shape)


Rows kept for ensemble window: (673000, 140)


In [28]:
# последний момент факта, из которого делаем прогноз
inference_ts = work_df["timestamp"].max()
test_model_df = work_df.loc[work_df["timestamp"] == inference_ts, feature_cols].copy()

print("Test rows:", test_model_df.shape)


Test rows: (1000, 131)


## Time-based split


In [ ]:
train_model_df = train_model_df.sort_values("source_timestamp").copy()

valid_end = train_model_df["source_timestamp"].max()
valid_start = valid_end - pd.Timedelta(days=VALID_DAYS)
calib_start = valid_end - pd.Timedelta(hours=12)  

fit_df = train_model_df[train_model_df["source_timestamp"] < valid_start].copy()
valid_early_df = train_model_df[
    (train_model_df["source_timestamp"] >= valid_start) &
    (train_model_df["source_timestamp"] < calib_start)
].copy()
valid_calib_df = train_model_df[train_model_df["source_timestamp"] >= calib_start].copy()

print("fit:", fit_df.shape)
print("valid_early:", valid_early_df.shape)
print("valid_calib:", valid_calib_df.shape)


fit: (624000, 140)
valid_early: (24000, 140)
valid_calib: (25000, 140)


In [30]:
X_fit = fit_df[feature_cols].copy()
y_fit = fit_df[FUTURE_TARGET_COLS].copy()

X_valid = valid_early_df[feature_cols].copy()
y_valid = valid_early_df[FUTURE_TARGET_COLS].copy()

X_calib = valid_calib_df[feature_cols].copy()
y_calib = valid_calib_df[FUTURE_TARGET_COLS].copy()

X_test = test_model_df[feature_cols].copy()

In [31]:
categorical_features = ["route_id"]
numeric_features = [col for col in feature_cols if col not in categorical_features]

all_route_categories = work_df["route_id"].cat.categories
for frame in [X_fit, X_valid, X_calib, X_test]:
    frame["route_id"] = pd.Categorical(frame["route_id"], categories=all_route_categories)

print("Categorical features:", categorical_features)
print("Numeric features:", len(numeric_features))


Categorical features: ['route_id']
Numeric features: 130


## Ансамбль моделей и blending
Ниже обучаются несколько моделей на одном и том же наборе фич, затем их прогнозы калибруются на `valid_calib_df` и смешиваются по весам, найденным на calibration slice.

In [ ]:
def wape_plus_rbias_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    denom = y_true.sum()
    if denom == 0:
        return np.nan

    wape = np.abs(y_pred - y_true).sum() / denom
    rbias = np.abs(y_pred.sum() / denom - 1.0)
    return float(wape + rbias)


def get_recent_subset(df: pd.DataFrame, train_days: int) -> pd.DataFrame:
    cutoff = df["source_timestamp"].max() - pd.Timedelta(days=train_days)
    return df.loc[df["source_timestamp"] >= cutoff].copy()


def get_time_decay_weights(df: pd.DataFrame, decay_days: float | None) -> np.ndarray | None:
    if decay_days is None:
        return None
    age_days = (
        (df["source_timestamp"].max() - df["source_timestamp"])
        .dt.total_seconds()
        .div(24 * 3600)
    )
    return np.exp(-age_days / decay_days)


def build_ridge_model(alpha: float):
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
            ("scaler", StandardScaler(with_mean=False)),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features),
        ],
        remainder="drop",
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", Ridge(alpha=alpha)),
        ]
    )

def prepare_catboost_frame(X: pd.DataFrame) -> pd.DataFrame:
    X_cb = X.copy()
    if "route_id" in X_cb.columns:
        X_cb["route_id"] = X_cb["route_id"].astype(str)
    return X_cb


def add_chain_features(
    X_base: pd.DataFrame,
    prev_targets_df: pd.DataFrame,
    step_idx: int,
    chain_depth: int | None = None,
) -> pd.DataFrame:
    """
    Для модели step_idx добавляет в признаки предыдущие target_step_*
    (в train/valid для fit используем истинные future targets).
    """
    X_aug = X_base.copy()

    start_step = 1 if chain_depth is None else max(1, step_idx - chain_depth)

    for prev_step in range(start_step, step_idx):
        prev_col = FUTURE_TARGET_COLS[prev_step - 1]
        X_aug[f"chain_prev_step_{prev_step}"] = prev_targets_df[prev_col].astype("float32").to_numpy()

    if "route_id" in X_aug.columns:
        X_aug["route_id"] = pd.Categorical(X_aug["route_id"], categories=all_route_categories)

    return X_aug


def rollout_chain_predictions(
    step_models: dict,
    X_base: pd.DataFrame,
    chain_depth: int | None = None,
) -> pd.DataFrame:
    """
    Последовательный inference:
    step_1 -> step_2 uses pred(step_1) -> step_3 uses pred(step_1, step_2) -> ...
    """
    pred_df = pd.DataFrame(index=X_base.index, columns=FUTURE_TARGET_COLS, dtype="float32")

    for step_idx, target_col in enumerate(FUTURE_TARGET_COLS, start=1):
        X_aug = X_base.copy()

        start_step = 1 if chain_depth is None else max(1, step_idx - chain_depth)

        for prev_step in range(start_step, step_idx):
            prev_col = FUTURE_TARGET_COLS[prev_step - 1]
            X_aug[f"chain_prev_step_{prev_step}"] = pred_df[prev_col].astype("float32").to_numpy()

        if "route_id" in X_aug.columns:
            X_aug["route_id"] = pd.Categorical(X_aug["route_id"], categories=all_route_categories)

        pred_df[target_col] = np.clip(step_models[target_col].predict(X_aug), 0, None).astype("float32")

    return pred_df


model_fit_pred_dfs = {}
model_valid_pred_dfs = {}
model_calib_pred_dfs = {}
model_test_pred_dfs = {}
model_scores = []

for spec in ACTIVE_MODEL_SPECS:
    spec_name = spec["name"]
    spec_kind = spec["kind"]

    fit_subset = get_recent_subset(fit_df, spec["train_days"])

    model_train_cap = spec.get("max_train_rows", MAX_TRAIN_ROWS)
    if len(fit_subset) > model_train_cap:
        fit_subset = fit_subset.sort_values("source_timestamp").tail(model_train_cap).copy()

    X_fit_m = fit_subset[feature_cols].copy()
    y_fit_m = fit_subset[FUTURE_TARGET_COLS].copy()

    if "route_id" in X_fit_m.columns:
        X_fit_m["route_id"] = pd.Categorical(X_fit_m["route_id"], categories=all_route_categories)

    sample_weight = get_time_decay_weights(fit_subset, spec.get("decay_days"))

    if spec_kind == "catboost":
        X_fit_cb = prepare_catboost_frame(X_fit_m)
        X_valid_cb = prepare_catboost_frame(X_valid)
        X_calib_cb = prepare_catboost_frame(X_calib)
        X_test_cb = prepare_catboost_frame(X_test)
        X_fit_full_cb = prepare_catboost_frame(X_fit)

    print(f"\n=== Training {spec_name} ===")
    print("fit subset:", fit_subset.shape, "| valid:", valid_early_df.shape, "| calib:", valid_calib_df.shape)

    # -----------------------------
    # CASE 1: обычный chain-LGBM
    # -----------------------------
    if spec_kind == "lgbm_chain":
        chain_depth = spec.get("chain_depth", None)
        step_models = {}

        for step_idx, target_col in enumerate(FUTURE_TARGET_COLS, start=1):
            X_fit_aug = add_chain_features(X_fit_m, y_fit_m, step_idx, chain_depth=chain_depth)
            X_valid_aug = add_chain_features(X_valid, y_valid, step_idx, chain_depth=chain_depth)

            model = LGBMRegressor(**spec["params"])
            model.fit(
                X_fit_aug,
                y_fit_m[target_col],
                sample_weight=sample_weight,
                eval_set=[(X_valid_aug, y_valid[target_col])],
                eval_metric="l1",
                categorical_feature=categorical_features,
                callbacks=[
                    early_stopping(stopping_rounds=100, verbose=False),
                    log_evaluation(period=0),
                ],
            )
            step_models[target_col] = model

        fit_pred = rollout_chain_predictions(step_models, X_fit, chain_depth=chain_depth)
        valid_pred = rollout_chain_predictions(step_models, X_valid, chain_depth=chain_depth)
        calib_pred = rollout_chain_predictions(step_models, X_calib, chain_depth=chain_depth)
        test_pred = rollout_chain_predictions(step_models, X_test, chain_depth=chain_depth)

    # -----------------------------
    # CASE 2: обычный LGBM / Ridge
    # -----------------------------
    else:
        fit_pred = pd.DataFrame(index=fit_df.index, columns=FUTURE_TARGET_COLS, dtype="float32")
        valid_pred = pd.DataFrame(index=valid_early_df.index, columns=FUTURE_TARGET_COLS, dtype="float32")
        calib_pred = pd.DataFrame(index=valid_calib_df.index, columns=FUTURE_TARGET_COLS, dtype="float32")
        test_pred = pd.DataFrame(index=test_model_df.index, columns=FUTURE_TARGET_COLS, dtype="float32")

        for target_col in FUTURE_TARGET_COLS:
            if spec_kind == "lgbm":
                model = LGBMRegressor(**spec["params"])
                model.fit(
                    X_fit_m,
                    y_fit_m[target_col],
                    sample_weight=sample_weight,
                    eval_set=[(X_valid, y_valid[target_col])],
                    eval_metric="l1",
                    categorical_feature=categorical_features,
                    callbacks=[
                        early_stopping(stopping_rounds=100, verbose=False),
                        log_evaluation(period=0),
                    ],
                )
            elif spec_kind == "catboost":
                model = CatBoostRegressor(**spec["params"])
                model.fit(
                    X_fit_cb,
                    y_fit_m[target_col],
                    sample_weight=sample_weight,
                    eval_set=(X_valid_cb, y_valid[target_col]),
                    cat_features=categorical_features,
                    use_best_model=True,
                    early_stopping_rounds=100,
                    verbose=False,
                )
            elif spec_kind == "ridge":
                model = build_ridge_model(alpha=spec["alpha"])
                if sample_weight is not None:
                    model.fit(X_fit_m, y_fit_m[target_col], model__sample_weight=sample_weight)
                else:
                    model.fit(X_fit_m, y_fit_m[target_col])
            else:
                raise ValueError(f"Unknown model kind: {spec_kind}")

            if spec_kind == "catboost":
                fit_pred[target_col] = np.clip(model.predict(X_fit_full_cb), 0, None).astype("float32")
                valid_pred[target_col] = np.clip(model.predict(X_valid_cb), 0, None).astype("float32")
                calib_pred[target_col] = np.clip(model.predict(X_calib_cb), 0, None).astype("float32")
                test_pred[target_col] = np.clip(model.predict(X_test_cb), 0, None).astype("float32")
            else:
                fit_pred[target_col] = np.clip(model.predict(X_fit), 0, None).astype("float32")
                valid_pred[target_col] = np.clip(model.predict(X_valid), 0, None).astype("float32")
                calib_pred[target_col] = np.clip(model.predict(X_calib), 0, None).astype("float32")
                test_pred[target_col] = np.clip(model.predict(X_test), 0, None).astype("float32")

    horizon_scales = {}
    for target_col in FUTURE_TARGET_COLS:
        pred_sum = calib_pred[target_col].sum()
        true_sum = y_calib[target_col].sum()
        scale = 1.0 if pred_sum == 0 else float(true_sum / pred_sum)
        horizon_scales[target_col] = scale

        fit_pred[target_col] *= scale
        valid_pred[target_col] *= scale
        calib_pred[target_col] *= scale
        test_pred[target_col] *= scale

    model_fit_pred_dfs[spec_name] = fit_pred
    model_valid_pred_dfs[spec_name] = valid_pred
    model_calib_pred_dfs[spec_name] = calib_pred
    model_test_pred_dfs[spec_name] = test_pred

    model_scores.append({
        "model": spec_name,
        "kind": spec_kind,
        "train_days": spec["train_days"],
        "chain_depth": spec.get("chain_depth", np.nan),
        "calib_score": wape_plus_rbias_score(y_calib.to_numpy().ravel(), calib_pred.to_numpy().ravel()),
        "valid_score": wape_plus_rbias_score(y_valid.to_numpy().ravel(), valid_pred.to_numpy().ravel()),
        "mean_scale": np.mean(list(horizon_scales.values())),
    })

score_table = pd.DataFrame(model_scores).sort_values("calib_score").reset_index(drop=True)
display(score_table)

print("Stored predictions for models:", list(model_calib_pred_dfs.keys()))


=== Training lgb_poisson_7d ===
fit subset: (337000, 140) | valid: (24000, 140) | calib: (25000, 140)

=== Training lgb_poisson_9d ===
fit subset: (433000, 140) | valid: (24000, 140) | calib: (25000, 140)

=== Training lgb_mae_7d ===
fit subset: (337000, 140) | valid: (24000, 140) | calib: (25000, 140)

=== Training lgb_poisson_5d ===
fit subset: (241000, 140) | valid: (24000, 140) | calib: (25000, 140)

=== Training lgb_poisson_14d ===
fit subset: (624000, 140) | valid: (24000, 140) | calib: (25000, 140)

=== Training lgb_mae_14d ===
fit subset: (624000, 140) | valid: (24000, 140) | calib: (25000, 140)

=== Training cat_poisson_14d ===
fit subset: (624000, 140) | valid: (24000, 140) | calib: (25000, 140)

=== Training ridge_7d ===
fit subset: (337000, 140) | valid: (24000, 140) | calib: (25000, 140)

=== Training ridge_7d_alpha_20 ===
fit subset: (337000, 140) | valid: (24000, 140) | calib: (25000, 140)


,model,kind,train_days,chain_depth,calib_score,valid_score,mean_scale
0,lgb_mae_14d,lgbm,14,NaN,0.330504,0.329383,1.037392
1,lgb_poisson_14d,lgbm,14,NaN,0.330738,0.327870,1.004017
2,cat_poisson_14d,catboost,14,NaN,0.331136,0.330930,1.041315
3,lgb_poisson_9d,lgbm,9,NaN,0.331282,0.328533,1.001502
4,lgb_poisson_7d,lgbm,7,NaN,0.332009,0.330083,1.000648
5,lgb_mae_7d,lgbm,7,NaN,0.332185,0.333438,1.035965
6,ridge_7d_alpha_20,ridge,7,NaN,0.332299,0.331575,1.002730
7,ridge_7d,ridge,7,NaN,0.332642,0.332400,1.004164
8,lgb_poisson_5d,lgbm,5,NaN,0.333118,0.335724,0.997106


Stored predictions for models: ['lgb_poisson_7d', 'lgb_poisson_9d', 'lgb_mae_7d', 'lgb_poisson_5d', 'lgb_poisson_14d', 'lgb_mae_14d', 'cat_poisson_14d', 'ridge_7d', 'ridge_7d_alpha_20']


In [ ]:
from itertools import combinations

def wape_plus_rbias_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    denom = y_true.sum()
    if denom == 0:
        return np.nan

    wape = np.abs(y_pred - y_true).sum() / denom
    rbias = np.abs(y_pred.sum() / denom - 1.0)
    return float(wape + rbias)

def generate_blend_weights(model_names, step=0.1):
    n = len(model_names)
    units = int(round(1.0 / step)) 
    for bars in combinations(range(units + n - 1), n - 1):
        parts = []
        prev = -1
        for b in bars:
            parts.append(b - prev - 1)
            prev = b
        parts.append(units + n - 1 - prev - 1)

        weights = np.array(parts, dtype=np.float32) * step
        yield dict(zip(model_names, weights))


def blend_prediction_dict_single_col(pred_dict, weights, col):
    base_name = next(iter(weights.keys()))
    blended = pred_dict[base_name][[col]].copy() * weights[base_name]
    for name, weight in list(weights.items())[1:]:
        blended = blended + pred_dict[name][[col]] * weight
    return blended.astype("float32")


def blend_prediction_dict_multi_col(pred_dict, per_horizon_weights, columns):
    blended = pd.DataFrame(index=next(iter(pred_dict.values())).index, columns=columns, dtype="float32")
    for col in columns:
        weights = per_horizon_weights[col]
        blended[col] = blend_prediction_dict_single_col(pred_dict, weights, col)[col]
    return blended.astype("float32")

# Вариант А: все модели из словаря
# model_names = list(model_calib_pred_dfs.keys())

# Вариант Б: руками выбрать лучшие
model_names = ['lgb_poisson_7d', 
               'lgb_poisson_9d', 
               'lgb_mae_7d', 
               'lgb_poisson_5d', 
               'lgb_poisson_14d', 
               'lgb_mae_14d', 
               'cat_poisson_14d', 
               'ridge_7d', 
               'ridge_7d_alpha_20']

model_names = [m for m in model_names if m in model_calib_pred_dfs]

print("Models used in per-horizon blend:", model_names)

best_score_per_horizon = {}
best_weights_per_horizon = {}

for col in FUTURE_TARGET_COLS:
    if len(model_names) == 1:
        best_weights_per_horizon[col] = {model_names[0]: 1.0}
        best_score_per_horizon[col] = wape_plus_rbias_score(
            y_calib[[col]].to_numpy().ravel(),
            model_calib_pred_dfs[model_names[0]][[col]].to_numpy().ravel()
        )
        continue

    best_score = None
    best_weights = None

    for weights in generate_blend_weights(model_names):
        calib_blend_col = blend_prediction_dict_single_col(model_calib_pred_dfs, weights, col)
        score = wape_plus_rbias_score(
            y_calib[[col]].to_numpy().ravel(),
            calib_blend_col.to_numpy().ravel()
        )

        if (best_score is None) or (score < best_score):
            best_score = score
            best_weights = weights
            print("Best blend weights (intermittent):")
            display(pd.Series(best_weights).sort_values(ascending=False))
            print("Best calibration score (intermittent):", round(best_score, 6))

    best_score_per_horizon[col] = best_score
    best_weights_per_horizon[col] = best_weights


print("Best per-horizon calibration scores:")
display(pd.Series(best_score_per_horizon).round(6))

print("Best weights per horizon:")
display(pd.DataFrame(best_weights_per_horizon).T.fillna(0.0))


fit_pred_df = blend_prediction_dict_multi_col(model_fit_pred_dfs, best_weights_per_horizon, FUTURE_TARGET_COLS).clip(lower=0)
valid_pred_df = blend_prediction_dict_multi_col(model_valid_pred_dfs, best_weights_per_horizon, FUTURE_TARGET_COLS).clip(lower=0)
calib_pred_df = blend_prediction_dict_multi_col(model_calib_pred_dfs, best_weights_per_horizon, FUTURE_TARGET_COLS).clip(lower=0)
test_pred_df = blend_prediction_dict_multi_col(model_test_pred_dfs, best_weights_per_horizon, FUTURE_TARGET_COLS).clip(lower=0)

display(test_pred_df.head())



overall_calib_score = wape_plus_rbias_score(
    y_calib.to_numpy().ravel(),
    calib_pred_df.to_numpy().ravel()
)

overall_valid_score = wape_plus_rbias_score(
    y_valid.to_numpy().ravel(),
    valid_pred_df.to_numpy().ravel()
)

print("Overall per-horizon blend calib score:", round(overall_calib_score, 6))
print("Overall per-horizon blend valid score:", round(overall_valid_score, 6))

Models used in per-horizon blend: ['lgb_poisson_7d', 'lgb_poisson_9d', 'lgb_mae_7d', 'lgb_poisson_5d', 'lgb_poisson_14d', 'lgb_mae_14d', 'cat_poisson_14d', 'ridge_7d', 'ridge_7d_alpha_20']
Best blend weights (intermittent):


ridge_7d_alpha_20    1.0
lgb_poisson_7d       0.0
lgb_poisson_9d       0.0
lgb_mae_7d           0.0
lgb_poisson_5d       0.0
lgb_poisson_14d      0.0
lgb_mae_14d          0.0
cat_poisson_14d      0.0
ridge_7d             0.0
dtype: float32

Best calibration score (intermittent): 0.262016
Best blend weights (intermittent):


ridge_7d_alpha_20    0.9
ridge_7d             0.1
lgb_poisson_7d       0.0
lgb_poisson_9d       0.0
lgb_mae_7d           0.0
lgb_poisson_5d       0.0
lgb_poisson_14d      0.0
lgb_mae_14d          0.0
cat_poisson_14d      0.0
dtype: float32

Best calibration score (intermittent): 0.262009
Best blend weights (intermittent):


...

target_step_1    0.254864
target_step_2    0.316308
target_step_3    0.333235
target_step_4    0.341644
target_step_5    0.345036
target_step_6    0.346902
target_step_7    0.345647
target_step_8    0.344964
dtype: float64

Best weights per horizon:


,lgb_poisson_7d,lgb_poisson_9d,lgb_mae_7d,lgb_poisson_5d,lgb_poisson_14d,lgb_mae_14d,cat_poisson_14d,ridge_7d,ridge_7d_alpha_20
target_step_1,0.0,0.0,0.0,0.0,0.2,0.1,0.5,0.2,0.0
target_step_2,0.0,0.0,0.0,0.0,0.3,0.2,0.3,0.2,0.0
target_step_3,0.0,0.0,0.0,0.0,0.4,0.1,0.2,0.3,0.0
target_step_4,0.0,0.0,0.0,0.0,0.2,0.3,0.2,0.3,0.0
target_step_5,0.0,0.0,0.0,0.0,0.3,0.3,0.1,0.2,0.1
target_step_6,0.0,0.0,0.0,0.0,0.3,0.3,0.2,0.2,0.0
target_step_7,0.0,0.0,0.0,0.0,0.3,0.5,0.0,0.1,0.1
target_step_8,0.0,0.0,0.0,0.0,0.2,0.5,0.0,0.0,0.3


,target_step_1,target_step_2,target_step_3,target_step_4,target_step_5,target_step_6,target_step_7,target_step_8
816,197474.312500,172431.781250,188838.218750,185274.843750,180014.765625,173146.812500,171705.640625,169212.125000
1633,95557.648438,94091.812500,75811.593750,74270.507812,81461.093750,76868.562500,79743.617188,79443.929688
2450,20021.824219,37798.226562,50202.738281,59710.632812,59450.832031,62678.320312,66721.031250,66402.328125
3267,99347.546875,211046.250000,168995.375000,149409.531250,144486.609375,135474.875000,124582.773438,123568.031250
4084,276263.437500,359082.843750,339825.531250,339965.093750,355763.812500,353702.312500,377100.750000,379334.625000


Overall per-horizon blend calib score: 0.328581
Overall per-horizon blend valid score: 0.326648


## Метрики


In [34]:
class WapePlusRbias:
    """Calculates as WAPE + Relative Bias."""

    @property
    def name(self) -> str:
        """Возвращает имя метрики."""
        return "wape_plus_rbias"

    def calculate(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Рассчитывает значение метрики."""
        wape = (np.abs(y_pred - y_true)).sum() / y_true.sum()
        rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
        return wape + rbias
            

metric = WapePlusRbias()

In [35]:
print('Метрики на обучающем срезе (blended):')
display(np.round(metric.calculate(y_fit.loc[fit_pred_df.index], fit_pred_df), 4))

print('Общая метрика на обучающем срезе:')
print(f'{metric.calculate(y_fit.loc[fit_pred_df.index].to_numpy().flatten(), fit_pred_df.to_numpy().flatten()):.4f}')


Метрики на обучающем срезе (blended):


target_step_1    0.2382
target_step_2    0.2911
target_step_3    0.3103
target_step_4    0.3174
target_step_5    0.3224
target_step_6    0.3290
target_step_7    0.3308
target_step_8    0.3324
dtype: float32

Общая метрика на обучающем срезе:
0.3088


In [36]:
print('Метрики на early-validation (blended):')
display(np.round(metric.calculate(y_valid, valid_pred_df), 4))

print('Общая метрика на early-validation:')
print(f'{metric.calculate(y_valid.to_numpy().flatten(), valid_pred_df.to_numpy().flatten()):.4f}')

print('\nМетрики на calibration slice (blended):')
display(np.round(metric.calculate(y_calib, calib_pred_df), 4))

print('Общая метрика на calibration slice:')
print(f'{metric.calculate(y_calib.to_numpy().flatten(), calib_pred_df.to_numpy().flatten()):.4f}')


Метрики на early-validation (blended):


target_step_1    0.2540
target_step_2    0.3188
target_step_3    0.3353
target_step_4    0.3360
target_step_5    0.3424
target_step_6    0.3440
target_step_7    0.3487
target_step_8    0.3475
dtype: float32

Общая метрика на early-validation:
0.3266

Метрики на calibration slice (blended):


target_step_1    0.2549
target_step_2    0.3163
target_step_3    0.3332
target_step_4    0.3416
target_step_5    0.3450
target_step_6    0.3469
target_step_7    0.3456
target_step_8    0.3450
dtype: float32

Общая метрика на calibration slice:
0.3286


## Конвертируем прогноз в нужный формат


In [37]:
# добавляем к прогнозу маршруты
test_pred_df['route_id'] = X_test['route_id']

# разворачиваем target_step_* в строки
forecast_df = test_pred_df.melt(
    id_vars="route_id",
    value_vars=[c for c in test_pred_df.columns if c.startswith("target_step_")],
    var_name="step",
    value_name="forecast"
)

# достаем номер шага из target_step_1, target_step_2, ...
forecast_df["step_num"] = forecast_df["step"].str.extract(r"(\d+)").astype(int)

# строим timestamp: каждый шаг = +30 минут от времени прогноза
forecast_df["timestamp"] = inference_ts + pd.to_timedelta(forecast_df["step_num"] * 30, unit="m")

# оставляем нужные столбцы
forecast_df = forecast_df[["route_id", "timestamp", "forecast"]].sort_values(
    ["route_id", "timestamp"]
).reset_index(drop=True)

forecast_df = test_df.merge(forecast_df, 'outer')[["id", "forecast"]]
forecast_df = forecast_df.rename(columns={"forecast": "y_pred"})

In [38]:
forecast_df.head()

,id,y_pred
0,3920,197474.312500
1,3921,172431.781250
2,3922,188838.218750
3,3923,185274.843750
4,3924,180014.765625


In [39]:
# проверяем, что все точки получены
assert forecast_df['id'].isna().sum() == 0

## Выгрузка CSV


In [40]:
submission_path =  f"submission_{TRACK}_add_catboost_to_blend.csv"
joined_path =  f"test_with_forecast_{TRACK}.csv"

forecast_df.to_csv(submission_path, index=False)

print("submission saved to:", submission_path)

submission saved to: submission_solo_add_catboost_to_blend.csv
